
![Workflow](Images/data_injestion_pipeline.png)

# Understanding Document Structure in RAG Pipelines

**Document Structure** is **not** the final stage or vector embedding itself. Instead, it happens at the **very beginning** of the pipeline during **Data Parsing**. 

---

## 1. What is Document Structure?

Document structure refers to understanding **how content is organized inside a file** before you slice it into chunks or convert it into vectors. 

Raw text loses its context and meaning if you ignore its underlying layout:

* **PDFs:** Have headers, footers, section numbers, multi-column layouts, and images.
* **Excel / SQL DBs:** Have structured relationships, tables, columns, and rows.
* **HTML Pages:** Have hierarchical DOM tags like `<h1>`, `<h2>`, `<article>`, and `<table>`.

---

## 2. Why Document Structure Matters Before Chunking & Embedding

1. **Enables Semantic Chunking:** If you blindly split a document every 500 characters, you risk cutting a table in half or detaching a heading from its supporting text. Recognizing structure allows you to split logically by natural boundaries (e.g., keeping complete paragraphs, sections, or table rows together).
2. **Generates Rich Metadata:** Extracting structural elements allows you to attach context metadata to each chunk in your Vector DB—such as `{"page_number": 4, "section_header": "3.1 Leave Policy", "file_type": "PDF"}`. This enables hybrid filtering during search.
3. **Improves Retrieval Quality:** Vector quality is directly dependent on the cleanliness of the input text. Preserving structure prevents garbage tokens or broken sentences from polluting your vector space.

---

## 3. Where It Fits in the Pipeline

```text
[Raw Files] ──> [Data Parsing & Document Structure] ──> [Chunks] ──> [Embedding Model] ──> [Vector DB]

![Workflow](Images/document_structure.png)
![Workflow](Images/pageContent_metadata.png)


In [ ]:
### Document structure
from langchain_core.documents import Document

In [2]:
doc= Document(
    page_content="This is the main text i am using to create rag",
    metadata={"source": "example.txt", 
              "pages": 1,
              "author": "Aryan Phanse", 
              "date": "2023-06-01"}
    )
doc

### metadata is usefull when we want to filter the documents based on certain criteria. For example, we can filter documents by author, date, or source. This can be useful when we have a large number of documents and we want to retrieve only those that match certain criteria.

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Aryan Phanse', 'date': '2023-06-01'}, page_content='This is the main text i am using to create rag')

In [3]:
## create a simple txt file
import os
os.makedirs("data/text_files", exist_ok=True)

In [6]:
sample_texts={
    "data/text_files/python_intro.txt": """
Python Programming Introduction

Python is a high-level, interpreted, general-purpose programming language created by Guido van Rossum and first released in 1991. It is popular because of its simple, readable syntax.

Key Features

* Easy to learn and use
* Interpreted and dynamically typed
* Object-oriented and supports multiple programming paradigms
* Cross-platform and open source
* Large collection of libraries

Applications

Python is widely used in web development, data science, artificial intelligence, machine learning, automation, scientific computing, and cybersecurity.

Basic Example

python-
print("Hello, World!")


Common Concepts

* Variables and data types
* Operators
* Conditional statements
* Loops
* Functions
* Lists, tuples, sets, and dictionaries
* Modules and packages
* Object-oriented programming
* Exception and file handling

Conclusion

Python is a powerful and beginner-friendly language with a wide range of applications. Its simple syntax and extensive libraries make it especially useful for modern software development, AI, ML, and data science.


""",
}

for filepath,content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("sample file created successfully")

sample file created successfully


![Workflow](Images/document_loaders.png)

In [ ]:
### Reading text from the above file we created using TextLoader
from langchain_community.document_loaders import TextLoader
loader= TextLoader("data/text_files/python_intro.txt",
                    encoding="utf-8",
                    show_progress=True)
loader
